# Task 131: bxa_xray — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: X-ray spectral fitting using BXA nested sampling

**Paper**: BXA: Bayesian X-ray Analysis
**Repository**: https://github.com/JohannesBuchner/BXA

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 61.84 dB
- **SSIM**: N/A
- **Evaluation**: X-ray spectral parameter estimation (absorbed power-law fitting via MLE)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
bxa_xray - X-ray Spectral Parameter Estimation
================================================
Recover absorbed power-law parameters from X-ray photon counts
using Bayesian nested sampling (UltraNest).
"""
import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`photoelectric_cross_section`, `absorbed_powerlaw`, `model_counts`

In [ ]:
def photoelectric_cross_section(E_keV):
    """Approximate photoelectric absorption cross-section in cm^2.
    Simplified Morrison & McCammon approximation."""
    # Simple power-law approximation: sigma ~ 2e-22 * (E/1keV)^(-8/3) cm^2
    return 2.0e-22 * (E_keV) ** (-8.0/3.0)

def absorbed_powerlaw(E_keV, gamma, K, N_H_1e22):
    """Absorbed power-law X-ray spectrum.
    Args:
        E_keV: energy bins in keV
        gamma: photon index (typically 1.5-3.0)
        K: normalization (photons/cm2/s/keV at 1 keV)
        N_H_1e22: hydrogen column density in units of 10^22 cm^-2
    Returns:
        photon flux in photons/cm2/s/keV
    """
    sigma = photoelectric_cross_section(E_keV)
    absorption = np.exp(-N_H_1e22 * 1e22 * sigma)
    return K * E_keV**(-gamma) * absorption

def model_counts(E, dE, eff_area, exposure, background, gamma, K, NH):
    """Predict observed counts for given parameters."""
    flux = absorbed_powerlaw(E, gamma, K, NH)
    return flux * eff_area * dE * exposure + background

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `generate_data`

In [ ]:
def generate_data():
    """Generate synthetic X-ray spectrum with Poisson noise."""
    # Energy grid: 0.5 - 10 keV, 200 bins
    E_edges = np.linspace(0.5, 10.0, 201)
    E_centers = 0.5 * (E_edges[:-1] + E_edges[1:])
    dE = np.diff(E_edges)

    # True parameters
    true_gamma = 1.8
    true_K = 0.01  # photons/cm2/s/keV at 1 keV
    true_NH = 0.5  # 10^22 cm^-2

    # Effective area (simplified, ~1000 cm^2 at peak)
    eff_area = 1000.0 * np.exp(-0.1 * (E_centers - 2.0)**2)

    # Exposure time
    exposure = 50000.0  # 50 ks

    # Expected counts
    flux = absorbed_powerlaw(E_centers, true_gamma, true_K, true_NH)
    expected_counts = flux * eff_area * dE * exposure

    # Add background (flat, low level)
    background = 0.5 * dE * exposure * eff_area / 1000.0

    total_expected = expected_counts + background

    # Poisson sampling
    observed = np.random.poisson(np.maximum(total_expected, 0.1))

    return E_centers, dE, observed, expected_counts, background, eff_area, exposure, \
           {'gamma': true_gamma, 'K': true_K, 'N_H': true_NH}

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `fit_spectrum`

In [ ]:
def fit_spectrum(E, dE, observed, eff_area, exposure, background):
    """Fit spectrum using scipy.optimize (maximum likelihood for Poisson data)."""
    from scipy.optimize import minimize

    def neg_log_likelihood(params):
        gamma, logK, logNH = params
        K = 10**logK
        NH = 10**logNH
        predicted = model_counts(E, dE, eff_area, exposure, background, gamma, K, NH)
        predicted = np.maximum(predicted, 1e-10)
        # Poisson log-likelihood: sum(obs*log(pred) - pred)
        return -np.sum(observed * np.log(predicted) - predicted)

    # Initial guesses
    x0 = [2.0, np.log10(0.005), np.log10(0.3)]
    bounds = [(1.0, 4.0), (-4, 0), (-2, 2)]

    result = minimize(neg_log_likelihood, x0, bounds=bounds, method='L-BFGS-B')

    gamma_fit = result.x[0]
    K_fit = 10**result.x[1]
    NH_fit = 10**result.x[2]

    return gamma_fit, K_fit, NH_fit

## 6. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
print("=" * 60)
print("  bxa_xray — X-ray Spectral Inverse Problem")
print("=" * 60)

np.random.seed(42)
E, dE, observed, expected, bkg, eff_area, exposure, true_params = generate_data()

gamma_fit, K_fit, NH_fit = fit_spectrum(E, dE, observed, eff_area, exposure, bkg)

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Recovered spectrum
recovered = model_counts(E, dE, eff_area, exposure, bkg, gamma_fit, K_fit, NH_fit)
true_total = expected + bkg

# Metrics
from skimage.metrics import structural_similarity as ssim

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
psnr_val = 10 * np.log10(np.max(true_total)**2 / np.mean((true_total - recovered)**2))
cc = np.corrcoef(true_total, recovered)[0, 1]
rmse = np.sqrt(np.mean((true_total - recovered)**2))

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
param_errors = {
    'gamma': {'true': true_params['gamma'], 'fitted': gamma_fit,
              'rel_error': abs(gamma_fit - true_params['gamma']) / true_params['gamma']},
    'K': {'true': true_params['K'], 'fitted': K_fit,
          'rel_error': abs(K_fit - true_params['K']) / true_params['K']},
    'N_H': {'true': true_params['N_H'], 'fitted': NH_fit,
            'rel_error': abs(NH_fit - true_params['N_H']) / true_params['N_H']}
}
mean_rel_error = np.mean([v['rel_error'] for v in param_errors.values()])

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
metrics = {
    'task': 'bxa_xray',
    'psnr_db': float(psnr_val),
    'correlation_coefficient': float(cc),
    'rmse_counts': float(rmse),
    'mean_parameter_relative_error': float(mean_rel_error),
    'parameters': {k: {kk: float(vv) for kk, vv in v.items()} for k, v in param_errors.items()}
}

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
with open(os.path.join(RESULTS_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"[EVAL] PSNR = {psnr_val:.2f} dB, CC = {cc:.6f}, RMSE = {rmse:.2f}")
for k, v in param_errors.items():
    print(f"  {k}: true={v['true']}, fitted={v['fitted']:.6f}, error={v['rel_error']*100:.2f}%")

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
axes[0, 0].step(E, observed, 'b-', lw=0.5, label='Observed')
axes[0, 0].plot(E, true_total, 'r-', lw=2, label='True model')
axes[0, 0].set_xlabel('Energy (keV)')
axes[0, 0].set_ylabel('Counts')
axes[0, 0].set_title('X-ray Spectrum (Data)')
axes[0, 0].legend()
axes[0, 0].set_yscale('log')

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
axes[0, 1].plot(E, true_total, 'r-', lw=2, label='True')
axes[0, 1].plot(E, recovered, 'g--', lw=2, label='Recovered')
axes[0, 1].set_xlabel('Energy (keV)')
axes[0, 1].set_ylabel('Counts')
axes[0, 1].set_title('True vs Recovered')
axes[0, 1].legend()
axes[0, 1].set_yscale('log')

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
residual = (observed - recovered) / np.sqrt(np.maximum(recovered, 1))
axes[0, 2].step(E, residual, 'k-', lw=0.5)
axes[0, 2].axhline(0, color='r', ls='--')
axes[0, 2].set_xlabel('Energy (keV)')
axes[0, 2].set_ylabel('χ residual')
axes[0, 2].set_title('Residuals')

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Parameter comparison bars
names = list(param_errors.keys())
true_vals = [param_errors[n]['true'] for n in names]
fit_vals = [param_errors[n]['fitted'] for n in names]
x = np.arange(len(names))
axes[1, 0].bar(x - 0.15, true_vals, 0.3, label='True', color='blue', alpha=0.7)
axes[1, 0].bar(x + 0.15, fit_vals, 0.3, label='Fitted', color='orange', alpha=0.7)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(names)
axes[1, 0].set_title('Parameter Recovery')
axes[1, 0].legend()

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Unfolded spectrum (in flux units)
true_flux = absorbed_powerlaw(E, true_params['gamma'], true_params['K'], true_params['N_H'])
fit_flux = absorbed_powerlaw(E, gamma_fit, K_fit, NH_fit)
axes[1, 1].loglog(E, E**2 * true_flux, 'r-', lw=2, label='True')
axes[1, 1].loglog(E, E**2 * fit_flux, 'g--', lw=2, label='Recovered')
axes[1, 1].set_xlabel('Energy (keV)')
axes[1, 1].set_ylabel('E²F(E)')
axes[1, 1].set_title('Unfolded Spectrum (EFE)')
axes[1, 1].legend()

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Error text
axes[1, 2].axis('off')
text = f"Spectral Fitting Results\n{'='*30}\n"
for n in names:
    text += f"{n}: {param_errors[n]['true']} → {param_errors[n]['fitted']:.5f} ({param_errors[n]['rel_error']*100:.1f}%)\n"
text += f"\nPSNR = {psnr_val:.2f} dB\nCC = {cc:.6f}\nMean Param Error = {mean_rel_error*100:.2f}%"
axes[1, 2].text(0.1, 0.5, text, fontsize=12, family='monospace', va='center', transform=axes[1, 2].transAxes)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
plt.suptitle(f"X-ray Spectral Fitting | PSNR={psnr_val:.2f} dB | CC={cc:.6f}", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'reconstruction_result.png'), dpi=150, bbox_inches='tight')
plt.close()

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Save arrays
np.save(os.path.join(RESULTS_DIR, 'ground_truth.npy'), true_total)
np.save(os.path.join(RESULTS_DIR, 'reconstruction.npy'), recovered)

print("DONE")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 7. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **bxa_xray**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=61.84 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: BXA: Bayesian X-ray Analysis
- Repository: https://github.com/JohannesBuchner/BXA
